# 🌿 KrishiMitra - PlantDoc Fine-Tuning Notebook (Google Colab GPU)

This notebook guides you step-by-step through fine-tuning the **EfficientNet-B0 Crop Disease Classifier** on real-world **PlantDoc** leaf crops merged with PlantVillage.

---

### ⚡ Key Features in this Notebook:
1. **Direct Kaggle Dataset Download!** Downloads `nirmalsankalana/plantdoc-dataset` directly inside Colab via `kagglehub` (No manual upload needed!).
2. **No Retraining Required for PlantVillage!** Reuses your existing pre-trained `efficientnet_b0_disease.pt` weights.
3. **🔍 Dataset & Class Match Verification Script:** Runs `verify_merged_dataset.py` before training to verify 100% path and 38-class matching.

### ⚡ Step 1: Verify GPU Connection
Make sure GPU is enabled in Colab (**Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
import torch
print("==========================================")
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not selected! Please go to Runtime -> Change runtime type -> Select T4 GPU")
print("==========================================")

### 📦 Step 2: Unzip Codebase
Upload small `disease_detection_code.zip` (~1.7 MB) to Colab storage panel `/content/` and extract it.

In [ ]:
# Unzip Codebase
!unzip -q /content/disease_detection_code.zip -d /content/disease_detection
%cd /content/disease_detection
print("✅ Codebase unzipped successfully!")

### 🌐 Step 3: Download PlantVillage Dataset (Direct Cloud Download)
Downloads PlantVillage dataset directly inside Colab (takes ~1-2 minutes).

In [ ]:
import os
print("📥 Downloading PlantVillage dataset...")
!wget -q -O /content/plantvillage_dataset.zip https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
print("📦 Extracting PlantVillage...")
!unzip -q /content/plantvillage_dataset.zip -d /content/
os.makedirs("/content/disease_detection/datasets/raw/plantvillage", exist_ok=True)
!mv /content/PlantVillage-Dataset-master/raw/color /content/disease_detection/datasets/raw/plantvillage/color
print("✅ PlantVillage dataset setup complete!")

### 🍃 Step 4: Direct Download PlantDoc Dataset from Kaggle (`nirmalsankalana/plantdoc-dataset`)
Downloads the PlantDoc dataset directly into Colab using `kagglehub` without uploading any files from PC!

In [ ]:
import os
import shutil
import kagglehub

print("📥 Downloading PlantDoc dataset directly from Kaggle (nirmalsankalana/plantdoc-dataset)...")
try:
    kaggle_path = kagglehub.dataset_download("nirmalsankalana/plantdoc-dataset")
    print("✅ Downloaded from Kaggle to:", kaggle_path)
    
    target_dir = "/content/disease_detection/datasets/raw/plantdoc_classification"
    os.makedirs(target_dir, exist_ok=True)
    
    for item in os.listdir(kaggle_path):
        s = os.path.join(kaggle_path, item)
        d = os.path.join(target_dir, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
            
    print("✅ PlantDoc dataset downloaded & placed successfully!")
except Exception as e:
    print("⚠️ Kagglehub fallback (using direct clone):", e)
    !git clone https://github.com/edwardraff/plantdoc-dataset.git /content/disease_detection/datasets/raw/plantdoc_classification
    print("✅ PlantDoc cloned successfully!")

### 🛠️ Step 5: Install Python Dependencies

In [ ]:
!pip install -q -r requirements.txt

### 🔍 Step 6: RUN VERIFICATION SCRIPT (Classes & Datasets Match Check)
Runs `verify_merged_dataset.py` to confirm 100% path resolution and class mapping across all 38 classes before fine-tuning starts.

In [ ]:
# Run dataset & class matching verification script
!PYTHONPATH=. python verify_merged_dataset.py

### 🏆 Step 7: Load Pre-Trained PlantVillage Weights (Skip Base Retraining!)
If you uploaded your pre-trained `efficientnet_b0_disease.pt` to `/content/disease_detection/saved_models/`, it will be automatically detected here.

In [ ]:
import os
pretrained_weight = "saved_models/efficientnet_b0_disease.pt"

if os.path.exists(pretrained_weight):
    print(f"✅ Pre-trained PlantVillage weights found at: {pretrained_weight}")
    print("🔥 Ready to fine-tune directly on PlantDoc real-world dataset!")
else:
    print("ℹ️ Pre-trained weights not uploaded to saved_models/. Fine-tuning will start with ImageNet backbone initialization.")

### 🏋️ Step 8: Train / Fine-Tune Real-World Model (EfficientNet-B0)
Runs 2-stage fine-tuning:
- **Stage 1 (5 epochs):** Freeze backbone, train classification head.
- **Stage 2 (20 epochs):** Fine-tune last 3 backbone blocks with differential learning rate.

In [ ]:
!PYTHONPATH=. python classification/train_realworld.py --epochs 20 --stage1-epochs 5 --batch-size 32 --mix-ratio 0.15 --loss WeightedCrossEntropy --device cuda

### 📊 Step 9: Evaluate Fine-Tuned Model Performance
Generates detailed accuracy, precision, recall, and real-world evaluation report.

In [ ]:
!PYTHONPATH=. python classification/evaluate_realworld.py

### 💾 Step 10: Download Fine-Tuned Model Weights to PC

In [ ]:
from google.colab import files
import os

files_to_dl = [
    "saved_models/efficientnet_b0_realworld.pt",
    "outputs/classification/class_mapping.json",
    "outputs/classification/realworld_comparison_report.md"
]

for f_path in files_to_dl:
    if os.path.exists(f_path):
        try:
            files.download(f_path)
            print(f"⬇️ Download initiated for: {f_path}")
        except Exception as e:
            print(f"❌ Could not download {f_path}: {e}")